# Keras Training on Vertex AI (Minimal, Canonical)
---
##### **IMPORTANT NOTE**: Vertex AI managed training execution is handled separately in notebooks/04_vertex_ai_managed_training.ipynb

---
This notebook is the **first portfolio-grade artifact** in the repository.

Scope:
- Validate environment + permissions
- Train a minimal Keras model
- Save model artifacts locally and to GCS
- Keep complexity intentionally low

Non-goals (for now):
- Pipelines
- Hyperparameter tuning
- Distributed training
- CI/CD


## Imports & Configuration

In [ ]:
import os
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models


### Access Colab Keys

In [ ]:
# Get core GCP IDs

# Project IDs
from google.colab import userdata
CANONICAL_TEMPLATE_PROJECT_ID = userdata.get("CANONICAL_TEMPLATE_PROJECT_ID")

from google.colab import userdata
PROJECT_ID = userdata.get("GCP_PROJECT_ID")

# Bucket & Prefixes
from google.colab import userdata
GCS_BUCKET = userdata.get("GCS_BUCKET")

from google.colab import userdata
DATA_PREP_PREFIX = userdata.get("DATA_PREP_PREFIX")

from google.colab import userdata
DEPLOYMENT_PREFIX = userdata.get("DEPLOYMENT_PREFIX")

from google.colab import userdata
MLOPS_PREFIX = userdata.get("MLOPS_PREFIX")

from google.colab import userdata
TRAINING_PREFIX = userdata.get("TRAINING_PREFIX")

# GitHub
from google.colab import userdata
GITHUB_USER = userdata.get("GITHUB_USER")

from google.colab import userdata
GITHUB_AUTHOR_NAME = userdata.get("GITHUB_AUTHOR_NAME")

from google.colab import userdata
GITHUB_AUTHOR_EMAIL = userdata.get("GITHUB_AUTHOR_EMAIL")

## Environment Configuration

In [ ]:


# Define the region where Vertex AI operations execute
REGION = "us-central1"


# Assign the retrieved value to an OS environment variable
os.environ["CANONICAL_TEMPLATE_PROJECT_ID"] = CANONICAL_TEMPLATE_PROJECT_ID
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["GCS_BUCKET"] = GCS_BUCKET
os.environ["DATA_PREP_PREFIX"] = DATA_PREP_PREFIX
os.environ["TRAINING_PREFIX"] = TRAINING_PREFIX
os.environ["MLOPS_PREFIX"] = MLOPS_PREFIX
os.environ["DEPLOYMENT_PREFIX"] = DEPLOYMENT_PREFIX

# Validation
required_env = {
    "CANONICAL_TEMPLATE_PROJECT_ID": CANONICAL_TEMPLATE_PROJECT_ID,
    "PROJECT_ID": PROJECT_ID,
    "REGION": REGION,
    "GCS_BUCKET": GCS_BUCKET,
    "TRAINING_PREFIX": TRAINING_PREFIX,
    "DEPLOYMENT_PREFIX": DEPLOYMENT_PREFIX,
    "MLOPS_PREFIX": MLOPS_PREFIX,
    "DATA_PREP_PREFIX": DATA_PREP_PREFIX
}

missing = [k for k, v in required_env.items() if not v]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {missing}")
else : print("Evironment configuration complete.")

Evironment configuration complete.


## Local Artifacts Path

IMPORTANT CONTEXT

- artifacts/ is gitignored

- This is runtime scratch space

- It is not part of the canonical repo structure

- This mirrors how Vertex jobs separate source vs outputs

In [ ]:
# Local artifact paths (ephemeral, gitignored)
LOCAL_ARTIFACT_DIR = Path("../artifacts/training")
LOCAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MODEL_PATH = LOCAL_ARTIFACT_DIR / "keras_model"

# Path Validation
print("LOCAL_ARTIFACT_DIR:", LOCAL_ARTIFACT_DIR)
print("LOCAL_MODEL_PATH:", LOCAL_MODEL_PATH)

LOCAL_ARTIFACT_DIR: ../artifacts/training
LOCAL_MODEL_PATH: ../artifacts/training/keras_model


## Initialize Vertex AI

- **project**: Specifies the Google Cloud project ID where Vertex AI resources (like models, endpoints, and jobs) are or will be located.
- **location**: defines the Google Cloud region (e.g., us-central1 or europe-west4) where Vertex AI operations will run.
- **staging_bucket**: Defines a defalut Cloud Storage Bucket URI that the SDK uses to stage temporary files and store artifacts generated during operations.

In [ ]:
# Validates IAM project wiring
from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=GCS_BUCKET,
)


## Dummy Dataset

This keeps focus on platform mechanics.

In [ ]:
# Simple synthetic dataset
X_train = np.random.rand(1000, 10)
y_train = (np.sum(X_train, axis=1) > 5).astype(int)

X_val = np.random.rand(100, 10)
y_val = (np.sum(X_val, axis=1) > 5).astype(int)


## Minimal Keras Model

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

## Local Sanity Run (VERY important)

---



In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
)


Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.5312 - loss: 0.7336 - val_accuracy: 0.5000 - val_loss: 0.7039
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5245 - loss: 0.6991 - val_accuracy: 0.5200 - val_loss: 0.6889
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5935 - loss: 0.6837 - val_accuracy: 0.6600 - val_loss: 0.6797
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6010 - loss: 0.6770 - val_accuracy: 0.6900 - val_loss: 0.6723
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6820 - loss: 0.6647 - val_accuracy: 0.6900 - val_loss: 0.6625


## Save Model Artifacts

In [ ]:
LOCAL_MODEL_DIR = Path("../artifacts/training")
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)


LOCAL_MODEL_PATH = LOCAL_MODEL_DIR / "model.keras"
model.save(LOCAL_MODEL_PATH)


### Verification

In [ ]:
LOCAL_MODEL_PATH.exists()

True

### Authentication For accessing GCP services

**NOTE**: Uncomment to do actual project work. Comment to test the steup prior to project work.

In [ ]:
# from google.colab import auth
# auth.authenticate_user(project_id=PROJECT_ID)

## Derived Paths

In [ ]:
# GCS output paths (already complete URIs). This makes bucket usage explicit and readable
TRAINING_OUTPUT_URI = os.environ.get("TRAINING_PREFIX")

### Save Model Artifacts to GCS

- Uses TensorFlow-native save

- Writes directly to GCS

- Mirrors how Vertex training jobs emit artifacts

In [ ]:
# GCS_MODEL_URI = f"{TRAINING_OUTPUT_URI.rstrip('/')}/model.keras"
# model.save(GCS_MODEL_URI)

# print(f"Model saved to GCS at: {GCS_MODEL_URI}")


> Note: GCS save skipped in lab environment due to IAM restrictions.
> Local artifact save is the authoritative output for Course 9.


## Notes

This notebook intentionally keeps scope minimal.

What this proves:
- Environment variables are wired correctly
- Keras training works in the Vertex context
- Artifacts are produced deterministically
- This notebook validates local → GCS model flow only.
- No Vertex jobs are launched here.
- Artifacts here are scaffolding for Course 10+.
- This notebook is safe to rerun.

